In [ ]:
import numpy as np

In [ ]:
NUM_NODE = 21
INWARD_ORI_INDEX = [(0, 1), (1, 2), (2, 3), (3, 4),
                    (0, 5), (5, 6), (6, 7), (7, 8),
                    (5, 9), (9, 10), (10, 11), (11, 12),
                    (9, 13), (13, 14), (14, 15), (15, 16),
                    (0, 17), (13, 17), (17, 18), (18, 19), (19, 20)]

INWARD = [(i, j) for (i, j) in INWARD_ORI_INDEX]
OUTWARD = [(j, i) for (i, j) in INWARD_ORI_INDEX]
NEIGHBOR = INWARD + OUTWARD

In [ ]:
class GraphUtil:
    def __init__(self):
        pass
    
    def get_sgp_mat(self, 
                    num_in: int, 
                    num_out: int, 
                    link: list):
        A = np.zeros((num_in, num_out))
        for (i, j) in link:
            A[i, j] = 1
        A_norm = A / np.sum(A, axis=0, keepdims=True)
        
        return A_norm
    
    def edge2mat(self, 
                 link: list, 
                 num_node: int):
        A = np.zeros((num_node, num_node))
        for (i, j) in link:
            A[j, i] = 1
        
        return A
    
    def get_k_scale_graph(self, 
                          scale: int, 
                          A):
        if scale == 1:
            return A
        An = np.zeros_like(A)
        A_power = np.eye(A.shape[0])
        for k in range(scale):
            A_power = A_power @ A
            An += A_power
        An[An > 0] = 1

        return An
    
    def normalize_digraph(self, 
                          A):
        Dl = np.sum(A, 0)
        h, w = A.shape
        Dn = np.zeros((w, w))
        for i in range(w):
            if Dl[i] > 0:
                Dn[i, i] = Dl[i] ** (-1)
        AD = np.dot(A, Dn)

        return AD
    
    def get_spatial_graph(self, 
                          num_node: int, 
                          self_link: list, 
                          inward: list, 
                          outward: list):
        I = self.edge2mat(self_link, num_node)
        In = self.normalize_digraph(self.edge2mat(inward, num_node))
        Out = self.normalize_digraph(self.edge2mat(outward, num_node))
        A = np.stack((I, In, Out))

        return A
    
    def normalize_adjacency_matrix(self, 
                                   A):
        node_degrees = A.sum(-1)
        degs_inv_sqrt = np.power(node_degrees, -0.5)
        norm_degs_matrix = np.eye(len(node_degrees)) * degs_inv_sqrt
        
        return (norm_degs_matrix @ A @ norm_degs_matrix).astype(np.float32)


    def k_adjacency(self, 
                    A: np.ndarray, 
                    k: int, 
                    with_self: bool = False, 
                    self_factor: float = 1.0):
        assert isinstance(A, np.ndarray)
        I = np.eye(len(A), dtype=A.dtype)
        if k == 0:
            return I
        Ak = np.minimum(np.linalg.matrix_power(A + I, k), 1) \
        - np.minimum(np.linalg.matrix_power(A + I, k - 1), 1)
        if with_self:
            Ak += (self_factor * I)
        
        return Ak

    def get_multiscale_spatial_graph(self, 
                                     num_node: int, 
                                     self_link: list, 
                                     inward: list, 
                                     outward: list):
        I = self.edge2mat(self_link, num_node)
        A1 = self.edge2mat(inward, num_node)
        A2 = self.edge2mat(outward, num_node)
        A3 = self.k_adjacency(A1, 2)
        A4 = self.k_adjacency(A2, 2)
        A1 = self.normalize_digraph(A1)
        A2 = self.normalize_digraph(A2)
        A3 = self.normalize_digraph(A3)
        A4 = self.normalize_digraph(A4)
        A = np.stack((I, A1, A2, A3, A4))
        
        return A

    def get_uniform_graph(self, 
                          num_node: int, 
                          self_link: list, 
                          neighbor: list):
        A = self.normalize_digraph(self.edge2mat(neighbor + self_link, num_node))
        
        return A

In [ ]:
class Graph:
    def __init__(self, 
                 num_node: int = NUM_NODE,
                 self_link: list = [(i, i) for i in range(NUM_NODE)],
                 inward: list = INWARD,
                 outward: list = OUTWARD,
                 neighbor: list = NEIGHBOR,
                 labeling_mode: str = 'spatial'):
        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor
        self.A = self.get_adjacency_matrix1(num_node, self_link, inward, outward, labeling_mode) if labeling_mode is not None else None

    def get_adjacency_matrix1(self,
                             num_node: int,
                             self_link: list,
                             inward: list,
                             outward: list,
                             labeling_mode: str = None):
        if labeling_mode is None:
            return self.A
        elif labeling_mode == 'spatial':
            A = GraphUtil().get_spatial_graph(num_node, 
                                            self_link, 
                                            inward, 
                                            outward)
        else:
            raise ValueError()
        
        return A

    def get_adjacency_matrix(self,
                             labeling_mode: str = None):
        if labeling_mode is None:
            return self.A
        elif labeling_mode == 'spatial':
            A = GraphUtil().get_spatial_graph(self.num_node, 
                                            self.self_link, 
                                            self.inward, 
                                            self.outward)
        else:
            raise ValueError()
        return A